# <font color="red"> =========create directories for the BrainSMASH correlation analysis=========

In [1]:
import os
import glob
import shutil
import numpy as np
from correlation_functions import process_regional_metrics, average_multiple_dfs, get_rgb_color_vector, brainsmash_correlation, correlate_tsv_columns
import pandas as pd

In [2]:
working_dir = "/Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations"

In [3]:
cd $working_dir

/Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations


In [4]:
os.makedirs(f"{working_dir}/correlations", exist_ok=True)
# make directories for each of the 4 correlation analyses
os.makedirs(f"{working_dir}/correlations/regional_TA_fieldfraction", exist_ok=True)
os.makedirs(f"{working_dir}/correlations/seed_FC_fieldfraction", exist_ok=True)
os.makedirs(f"{working_dir}/correlations/DBM_fieldfraction", exist_ok=True)
os.makedirs(f"{working_dir}/correlations/struc_conn_fieldfraction", exist_ok=True)
os.makedirs(f"{working_dir}/correlations/DBM_struc_conn", exist_ok=True)

os.makedirs(f"{working_dir}/correlations/DBM_fieldfraction_stephanie", exist_ok=True)
os.makedirs(f"{working_dir}/correlations/DBM_struc_conn_stephanie", exist_ok=True)

os.makedirs(f"{working_dir}/correlations/DBM_fieldfraction_stephanie_smoothed", exist_ok=True)
os.makedirs(f"{working_dir}/correlations/DBM_struc_conn_stephanie_smoothed", exist_ok=True)

# make directories for each of the 4 metrics to be correlated with fieldfraction
os.makedirs(f"{working_dir}/metrics", exist_ok=True)

os.makedirs(f"{working_dir}/metrics/fieldfraction", exist_ok=True)

os.makedirs(f"{working_dir}/metrics/regional_TA", exist_ok=True)
os.makedirs(f"{working_dir}/metrics/seed_FC", exist_ok=True)
os.makedirs(f"{working_dir}/metrics/DBM", exist_ok=True)
os.makedirs(f"{working_dir}/metrics/struc_conn", exist_ok=True)

os.makedirs(f"{working_dir}/metrics/DBM_stephanie", exist_ok=True)
os.makedirs(f"{working_dir}/metrics/DBM_stephanie_smoothed", exist_ok=True)

# <font color="red"> =========define the PFF subjects=========

In [5]:
pff_subjects = [
    "sub-hM8362334086041083M6",
    "sub-hM836323993941035M4",
    "sub-hM8363224086001085M5",
    "sub-hM836373993921113M9",
    "sub-hM8362324086031084M5",
]

# <font color="#3e88c9ff"> =========remove the row corresponding to the seed region (CP) from the R_stru_tsv and R_positive_tsv files=========

In [6]:
# the structure conn has only an average from Allen connectivity, so we will just copy that file to the metrics/struc_conn directory

In [7]:
struc_conn_tsv_quant_dir = ("/Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/conn_mats")

In [8]:
# remove the row corresponding to the seed region (CP) from the R_stru_tsv and R_positive_tsv files
R_stru_positive_df = pd.read_csv(f"{struc_conn_tsv_quant_dir}/CP_avg_conn_right_positive_str_log10.tsv", sep="\t")
R_stru_positive_df = R_stru_positive_df[R_stru_positive_df["name"] != "R_Striatum dorsal region"]
# save the modified R_stru_positive_df to the metrics/struc_conn directory
R_stru_positive_df.to_csv(f"{struc_conn_tsv_quant_dir}/CP_avg_conn_right_positive_str_log10.tsv", sep="\t", index=False)

# <font color="#3e88c9ff"> =========process the struc_conn metric=========

In [9]:
all_stru_tsv = f"{struc_conn_tsv_quant_dir}/CP_avg_conn_all_str_log10.tsv"
R_stru_tsv = f"{struc_conn_tsv_quant_dir}/CP_avg_conn_right_str_log10.tsv"
R_positive_tsv = f"{struc_conn_tsv_quant_dir}/CP_avg_conn_right_positive_str_log10.tsv"

In [10]:
# copy the all_stru_tsv to the metrics/struc_conn directory
shutil.copy(all_stru_tsv, f"{working_dir}/metrics/struc_conn/CP_avg_conn_all_str_log10.tsv")
shutil.copy(R_stru_tsv, f"{working_dir}/metrics/struc_conn/CP_avg_conn_right_str_log10.tsv")
shutil.copy(R_positive_tsv, f"{working_dir}/metrics/struc_conn/CP_avg_conn_right_positive_str_log10.tsv")

'/Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/struc_conn/CP_avg_conn_right_positive_str_log10.tsv'

# <font color="#3e88c9ff"> =========process the fieldfraction data=========

In [11]:
LSFM_tsv_quant_dir = "/Volumes/LSFM/Blaze_quant_no_log_IAR_3_lvl_April_version"
algorithm = "th800"

LSFM_pff_files = [
    f"{LSFM_tsv_quant_dir}/sub-hM8362334086041083M6/tabular/{pff_subjects[0]}_sample-brain_acq-imaris_seg-mid_from-ABAv3_desc-th900_mergedsegstats.tsv", # was 900
    f"{LSFM_tsv_quant_dir}/sub-hM836323993941035M4/tabular/{pff_subjects[1]}_sample-brain_acq-imaris_seg-mid_from-ABAv3_desc-{algorithm}_mergedsegstats.tsv",
    f"{LSFM_tsv_quant_dir}/sub-hM8363224086001085M5/tabular/{pff_subjects[2]}_sample-brain_acq-imaris_seg-mid_from-ABAv3_desc-{algorithm}_mergedsegstats.tsv",
    f"{LSFM_tsv_quant_dir}/sub-hM836373993921113M9/tabular/{pff_subjects[3]}_sample-brain_acq-imaris_seg-mid_from-ABAv3_desc-{algorithm}_mergedsegstats.tsv",
    f"{LSFM_tsv_quant_dir}/sub-hM8362324086031084M5/tabular/{pff_subjects[4]}_sample-brain_acq-imaris_seg-mid_from-ABAv3_desc-{algorithm}_mergedsegstats.tsv",
]

print(f"{LSFM_pff_files}")

['/Volumes/LSFM/Blaze_quant_no_log_IAR_3_lvl_April_version/sub-hM8362334086041083M6/tabular/sub-hM8362334086041083M6_sample-brain_acq-imaris_seg-mid_from-ABAv3_desc-th900_mergedsegstats.tsv', '/Volumes/LSFM/Blaze_quant_no_log_IAR_3_lvl_April_version/sub-hM836323993941035M4/tabular/sub-hM836323993941035M4_sample-brain_acq-imaris_seg-mid_from-ABAv3_desc-th800_mergedsegstats.tsv', '/Volumes/LSFM/Blaze_quant_no_log_IAR_3_lvl_April_version/sub-hM8363224086001085M5/tabular/sub-hM8363224086001085M5_sample-brain_acq-imaris_seg-mid_from-ABAv3_desc-th800_mergedsegstats.tsv', '/Volumes/LSFM/Blaze_quant_no_log_IAR_3_lvl_April_version/sub-hM836373993921113M9/tabular/sub-hM836373993921113M9_sample-brain_acq-imaris_seg-mid_from-ABAv3_desc-th800_mergedsegstats.tsv', '/Volumes/LSFM/Blaze_quant_no_log_IAR_3_lvl_April_version/sub-hM8362324086031084M5/tabular/sub-hM8362324086031084M5_sample-brain_acq-imaris_seg-mid_from-ABAv3_desc-th800_mergedsegstats.tsv']


In [12]:
for pff_subject, pff_file in zip(pff_subjects, LSFM_pff_files):
    # i did not use f"{tsv_quant}/subject_..." becasue of the algorithm thing
    process_regional_metrics(pff_file,
                             output_dir=f"{working_dir}/metrics/fieldfraction",
                             output_tsv_stem=f"{pff_subject}_fieldfraction",
                             key_word="aSync+fieldfrac")

Right hemisphere tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/fieldfraction/sub-hM8362334086041083M6_fieldfraction_R.tsv
Right hemisphere tsv filtered by structural conn saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/fieldfraction/sub-hM8362334086041083M6_fieldfraction_R_stru_conn.tsv
Bilateral tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/fieldfraction/sub-hM8362334086041083M6_fieldfraction_bilateral.tsv
Right hemisphere tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/fieldfraction/sub-hM836323993941035M4_fieldfraction_R.tsv
Right hemisphere tsv filtered by structural conn saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/fieldfraction/sub-hM836323993941035M4_fieldfraction_R_stru_conn.tsv
Bilateral 

In [13]:
# get a list of the bilateral fieldfraction tsv files
fieldfraction_bilateral_tsv_files = glob.glob(f"{working_dir}/metrics/fieldfraction/sub-*_bilateral.tsv")

# get a list of R fieldfraction tsv files
fieldfraction_R_tsv_files = glob.glob(f"{working_dir}/metrics/fieldfraction/sub-*_R.tsv")

# get a list of the R postivie fieldfraction tsv files
fieldfraction_R_positive_tsv_files = glob.glob(f"{working_dir}/metrics/fieldfraction/sub-*_R_stru_conn.tsv")

In [14]:
# average the bilateral fieldfraction tsv files
average_multiple_dfs(fieldfraction_bilateral_tsv_files,
                     output_dir=f"{working_dir}/metrics/fieldfraction",
                     output_tsv_stem="average_fieldfraction_bilateral",
                     group_by_col="bilateral_region",
                     key_word="aSync+fieldfrac_bilateral")

average_multiple_dfs(fieldfraction_R_tsv_files,
                     output_dir=f"{working_dir}/metrics/fieldfraction",
                     output_tsv_stem="average_fieldfraction_R",
                     group_by_col="name",
                     key_word="aSync+fieldfrac")

average_multiple_dfs(fieldfraction_R_positive_tsv_files,
                     output_dir=f"{working_dir}/metrics/fieldfraction",
                     output_tsv_stem="average_fieldfraction_R_positive",
                     group_by_col="name",
                     key_word="aSync+fieldfrac")

Averaged tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/fieldfraction/average_fieldfraction_bilateral.tsv
Averaged tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/fieldfraction/average_fieldfraction_R.tsv
Averaged tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/fieldfraction/average_fieldfraction_R_positive.tsv


,name,aSync+fieldfrac
0,R_Somatomotor areas,1.023600
1,R_Somatosensory areas,0.871514
2,R_Auditory areas,0.628134
3,R_Visual areas,0.652930
4,R_Anterior cingulate area,1.025809
5,R_Prelimbic area,1.022342
6,R_Orbital area,0.518773
7,R_Agranular insular area,0.374882
8,R_Retrosplenial area,1.192026
9,R_Posterior parietal association areas,0.662168


# <font color="#3e88c9ff"> =========process the Regional_TA metric=========

In [285]:
TA_tsv_quant_dir = "/Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/FC_51_ROI_bilateral_interleaved/FC_average_51_ROI_outputdir/average_regional_TA"

In [286]:
for pff_subject in pff_subjects:
    print(f"Processing {pff_subject}")
    pff_file = f"{TA_tsv_quant_dir}/{pff_subject}/average_regional_TA.tsv"

    process_regional_metrics(pff_file,
                             output_dir=f"{working_dir}/metrics/regional_TA",
                             output_tsv_stem=f"{pff_subject}_regional_TA",
                             key_word="regional_TA")

Processing sub-hM8362334086041083M6
Right hemisphere tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/regional_TA/sub-hM8362334086041083M6_regional_TA_R.tsv
Right hemisphere tsv filtered by structural conn saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/regional_TA/sub-hM8362334086041083M6_regional_TA_R_stru_conn.tsv
Bilateral tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/regional_TA/sub-hM8362334086041083M6_regional_TA_bilateral.tsv
Processing sub-hM836323993941035M4
Right hemisphere tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/regional_TA/sub-hM836323993941035M4_regional_TA_R.tsv
Right hemisphere tsv filtered by structural conn saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/regional_TA/sub-hM83632

In [287]:
# get a list of the bilateral regional_TA tsv files
regional_TA_bilateral_tsv_files = glob.glob(f"{working_dir}/metrics/regional_TA/sub-*_bilateral.tsv")

# get a list of R regional_TA tsv files
regional_TA_R_tsv_files = glob.glob(f"{working_dir}/metrics/regional_TA/sub-*_R.tsv")

# get a list of the R postivie regional_TA tsv files
regional_TA_R_positive_tsv_files = glob.glob(f"{working_dir}/metrics/regional_TA/sub-*_R_stru_conn.tsv")

In [288]:
# average the bilateral regional_TA tsv files
average_multiple_dfs(regional_TA_bilateral_tsv_files,
                     output_dir=f"{working_dir}/metrics/regional_TA",
                     output_tsv_stem="average_regional_TA_bilateral",
                     group_by_col="bilateral_region",
                     key_word="regional_TA_bilateral")

average_multiple_dfs(regional_TA_R_tsv_files,
                     output_dir=f"{working_dir}/metrics/regional_TA",
                     output_tsv_stem="average_regional_TA_R",
                     group_by_col="name",
                     key_word="regional_TA")

average_multiple_dfs(regional_TA_R_positive_tsv_files,
                     output_dir=f"{working_dir}/metrics/regional_TA",
                     output_tsv_stem="average_regional_TA_R_positive",
                     group_by_col="name",
                     key_word="regional_TA")

Averaged tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/regional_TA/average_regional_TA_bilateral.tsv
Averaged tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/regional_TA/average_regional_TA_R.tsv
Averaged tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/regional_TA/average_regional_TA_R_positive.tsv


,name,regional_TA
0,R_Somatomotor areas,0.098785
1,R_Somatosensory areas,0.159873
2,R_Auditory areas,0.092680
3,R_Visual areas,0.088982
4,R_Anterior cingulate area,0.059593
5,R_Prelimbic area,0.031867
6,R_Orbital area,0.097875
7,R_Agranular insular area,0.063249
8,R_Retrosplenial area,0.079739
9,R_Posterior parietal association areas,0.047499


# <font color="#3e88c9ff"> =========process the seed_FC metric=========

In [289]:
seed_conn_tsv_quant_dir = "/Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/STRd_seed_based_analysis/dSTR_average_outputdir/seed_roi_z_values"

In [290]:
for pff_subject in pff_subjects:
    print(f"Processing {pff_subject}")
    pff_file = f"{seed_conn_tsv_quant_dir}/ses-01_{pff_subject}/seed_roi_z_values.tsv"

    process_regional_metrics(pff_file,
                             output_dir=f"{working_dir}/metrics/seed_FC",
                             output_tsv_stem=f"{pff_subject}_seed_FC",
                             key_word="mean_voxel_value")

Processing sub-hM8362334086041083M6
Right hemisphere tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/seed_FC/sub-hM8362334086041083M6_seed_FC_R.tsv
Right hemisphere tsv filtered by structural conn saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/seed_FC/sub-hM8362334086041083M6_seed_FC_R_stru_conn.tsv
Bilateral tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/seed_FC/sub-hM8362334086041083M6_seed_FC_bilateral.tsv
Processing sub-hM836323993941035M4
Right hemisphere tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/seed_FC/sub-hM836323993941035M4_seed_FC_R.tsv
Right hemisphere tsv filtered by structural conn saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/seed_FC/sub-hM836323993941035M4_seed_FC_R_stru_conn.tsv

In [291]:
# get a list of the bilateral seed_FC tsv files
seed_FC_bilateral_tsv_files = glob.glob(f"{working_dir}/metrics/seed_FC/sub-*_bilateral.tsv")

# get a list of R seed_FC tsv files
seed_FC_R_tsv_files = glob.glob(f"{working_dir}/metrics/seed_FC/sub-*_R.tsv")

# get a list of the R postivie seed_FC tsv files
seed_FC_R_positive_tsv_files = glob.glob(f"{working_dir}/metrics/seed_FC/sub-*_R_stru_conn.tsv")

In [292]:
# average the bilateral seed_FC tsv files
average_multiple_dfs(seed_FC_bilateral_tsv_files,
                     output_dir=f"{working_dir}/metrics/seed_FC",
                     output_tsv_stem="average_seed_FC_bilateral",
                     group_by_col="bilateral_region",
                     key_word="mean_voxel_value_bilateral")

average_multiple_dfs(seed_FC_R_tsv_files,
                     output_dir=f"{working_dir}/metrics/seed_FC",
                     output_tsv_stem="average_seed_FC_R",
                     group_by_col="name",
                     key_word="mean_voxel_value")

average_multiple_dfs(seed_FC_R_positive_tsv_files,
                     output_dir=f"{working_dir}/metrics/seed_FC",
                     output_tsv_stem="average_seed_FC_R_positive",
                     group_by_col="name",
                     key_word="mean_voxel_value")

Averaged tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/seed_FC/average_seed_FC_bilateral.tsv
Averaged tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/seed_FC/average_seed_FC_R.tsv
Averaged tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/seed_FC/average_seed_FC_R_positive.tsv


,name,mean_voxel_value
0,R_Somatomotor areas,0.061777
1,R_Somatosensory areas,0.060526
2,R_Auditory areas,0.053271
3,R_Visual areas,0.042321
4,R_Anterior cingulate area,0.059259
5,R_Prelimbic area,0.065039
6,R_Orbital area,0.081295
7,R_Agranular insular area,0.051379
8,R_Retrosplenial area,0.064953
9,R_Posterior parietal association areas,0.060188


# <font color="#3e88c9ff"> =========process the DBM metric=========

In [293]:
DBM_tsv_quant_dir = ("/Users/aeed/Documents/Work/M83_clearing"
                          "/Manuscript_analysis/dbm_output_original_resolution/dbm/jacobian/full/")

In [294]:
for pff_subject in pff_subjects:
    print(f"Processing {pff_subject}")
    pff_file = f"{DBM_tsv_quant_dir}/{pff_subject}_atrophy_map_roi_means.tsv"

    process_regional_metrics(pff_file,
                             output_dir=f"{working_dir}/metrics/DBM",
                             output_tsv_stem=f"{pff_subject}_DBM",
                             key_word="mean_voxel_value")

Processing sub-hM8362334086041083M6
Right hemisphere tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/DBM/sub-hM8362334086041083M6_DBM_R.tsv
Right hemisphere tsv filtered by structural conn saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/DBM/sub-hM8362334086041083M6_DBM_R_stru_conn.tsv
Bilateral tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/DBM/sub-hM8362334086041083M6_DBM_bilateral.tsv
Processing sub-hM836323993941035M4
Right hemisphere tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/DBM/sub-hM836323993941035M4_DBM_R.tsv
Right hemisphere tsv filtered by structural conn saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/DBM/sub-hM836323993941035M4_DBM_R_stru_conn.tsv
Bilateral tsv saved to:  /Users/aeed/Do

In [295]:
# get a list of the bilateral DBM tsv files
DBM_bilateral_tsv_files = glob.glob(f"{working_dir}/metrics/DBM/sub-*_bilateral.tsv")

# get a list of R DBM tsv files
DBM_R_tsv_files = glob.glob(f"{working_dir}/metrics/DBM/sub-*_R.tsv")

# get a list of the R postivie DBM tsv files
DBM_R_positive_tsv_files = glob.glob(f"{working_dir}/metrics/DBM/sub-*_R_stru_conn.tsv")

In [296]:
# average the bilateral DBM tsv files
average_multiple_dfs(DBM_bilateral_tsv_files,
                     output_dir=f"{working_dir}/metrics/DBM",
                     output_tsv_stem="average_DBM_bilateral",
                     group_by_col="bilateral_region",
                     key_word="mean_voxel_value_bilateral")

average_multiple_dfs(DBM_R_tsv_files,
                     output_dir=f"{working_dir}/metrics/DBM",
                     output_tsv_stem="average_DBM_R",
                     group_by_col="name",
                     key_word="mean_voxel_value")

average_multiple_dfs(DBM_R_positive_tsv_files,
                     output_dir=f"{working_dir}/metrics/DBM",
                     output_tsv_stem="average_DBM_R_positive",
                     group_by_col="name",
                     key_word="mean_voxel_value")

Averaged tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/DBM/average_DBM_bilateral.tsv
Averaged tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/DBM/average_DBM_R.tsv
Averaged tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/DBM/average_DBM_R_positive.tsv


,name,mean_voxel_value
0,R_Somatomotor areas,0.105120
1,R_Somatosensory areas,-0.065075
2,R_Auditory areas,0.586728
3,R_Visual areas,0.301146
4,R_Anterior cingulate area,-0.014002
5,R_Prelimbic area,-0.556011
6,R_Orbital area,-0.300403
7,R_Agranular insular area,0.360334
8,R_Retrosplenial area,0.019897
9,R_Posterior parietal association areas,0.623536


# <font color="#3e88c9ff"> =========process the DBM_stephanie metric=========


In [297]:
DBM_tsv_quant_dir = ("/Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/dbm_output_original_resolution/Stephanie_data/atrophy_maps")

In [298]:
for pff_subject in pff_subjects:
    print(f"Processing {pff_subject}")
    pff_file = f"{DBM_tsv_quant_dir}/{pff_subject}_atrophy_map_roi_means.tsv"

    process_regional_metrics(pff_file,
                             output_dir=f"{working_dir}/metrics/DBM_stephanie",
                             output_tsv_stem=f"{pff_subject}_DBM_stephanie",
                             key_word="mean_voxel_value")

Processing sub-hM8362334086041083M6
Right hemisphere tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/DBM_stephanie/sub-hM8362334086041083M6_DBM_stephanie_R.tsv
Right hemisphere tsv filtered by structural conn saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/DBM_stephanie/sub-hM8362334086041083M6_DBM_stephanie_R_stru_conn.tsv
Bilateral tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/DBM_stephanie/sub-hM8362334086041083M6_DBM_stephanie_bilateral.tsv
Processing sub-hM836323993941035M4
Right hemisphere tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/DBM_stephanie/sub-hM836323993941035M4_DBM_stephanie_R.tsv
Right hemisphere tsv filtered by structural conn saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/DBM_ste

In [299]:
# get a list of the bilateral DBM tsv files
DBM_bilateral_tsv_files = glob.glob(f"{working_dir}/metrics/DBM_stephanie/sub-*_bilateral.tsv")

# get a list of R DBM tsv files
DBM_R_tsv_files = glob.glob(f"{working_dir}/metrics/DBM_stephanie/sub-*_R.tsv")

# get a list of the R postivie DBM tsv files
DBM_R_positive_tsv_files = glob.glob(f"{working_dir}/metrics/DBM_stephanie/sub-*_R_stru_conn.tsv")

In [300]:

# average the bilateral DBM tsv files
average_multiple_dfs(DBM_bilateral_tsv_files,
                     output_dir=f"{working_dir}/metrics/DBM_stephanie",
                     output_tsv_stem="average_DBM_stephanie_bilateral",
                     group_by_col="bilateral_region",
                     key_word="mean_voxel_value_bilateral")

average_multiple_dfs(DBM_R_tsv_files,
                     output_dir=f"{working_dir}/metrics/DBM_stephanie",
                     output_tsv_stem="average_DBM_stephanie_R",
                     group_by_col="name",
                     key_word="mean_voxel_value")

average_multiple_dfs(DBM_R_positive_tsv_files,
                     output_dir=f"{working_dir}/metrics/DBM_stephanie",
                     output_tsv_stem="average_DBM_stephanie_R_positive",
                     group_by_col="name",
                     key_word="mean_voxel_value")

Averaged tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/DBM_stephanie/average_DBM_stephanie_bilateral.tsv
Averaged tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/DBM_stephanie/average_DBM_stephanie_R.tsv
Averaged tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/DBM_stephanie/average_DBM_stephanie_R_positive.tsv


,name,mean_voxel_value
0,R_Somatomotor areas,-9.795644
1,R_Somatosensory areas,-11.639556
2,R_Auditory areas,-8.938427
3,R_Visual areas,-10.091659
4,R_Anterior cingulate area,-10.993207
5,R_Prelimbic area,-9.145781
6,R_Orbital area,-9.659903
7,R_Agranular insular area,-9.628844
8,R_Retrosplenial area,-9.888497
9,R_Posterior parietal association areas,-11.464287


# <font color="#3e88c9ff"> =========process the DBM_stephanie_smoothed metric=========


In [25]:
DBM_tsv_quant_dir = ("/Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/dbm_output_original_resolution/Stephanie_data_smoothed/atrophy_maps")

In [26]:
DBM_tsv_quant_dir = ("/Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/dbm_output_original_resolution/Stephanie_data_smoothed/atrophy_maps")

In [27]:
for pff_subject in pff_subjects:
    print(f"Processing {pff_subject}")
    pff_file = f"{DBM_tsv_quant_dir}/{pff_subject}_atrophy_map_masked_roi_means.tsv"

    process_regional_metrics(pff_file,
                             output_dir=f"{working_dir}/metrics/DBM_stephanie_smoothed",
                             output_tsv_stem=f"{pff_subject}_DBM_stephanie_smoothed",
                             key_word="mean_voxel_value")

Processing sub-hM8362334086041083M6
Right hemisphere tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/DBM_stephanie_smoothed/sub-hM8362334086041083M6_DBM_stephanie_smoothed_R.tsv
Right hemisphere tsv filtered by structural conn saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/DBM_stephanie_smoothed/sub-hM8362334086041083M6_DBM_stephanie_smoothed_R_stru_conn.tsv
Bilateral tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/DBM_stephanie_smoothed/sub-hM8362334086041083M6_DBM_stephanie_smoothed_bilateral.tsv
Processing sub-hM836323993941035M4
Right hemisphere tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/DBM_stephanie_smoothed/sub-hM836323993941035M4_DBM_stephanie_smoothed_R.tsv
Right hemisphere tsv filtered by structural conn saved to:  /Users/aeed/Documents/Work/

In [28]:
# get a list of the bilateral DBM tsv files
DBM_bilateral_tsv_files = glob.glob(f"{working_dir}/metrics/DBM_stephanie_smoothed/sub-*_bilateral.tsv")

# get a list of R DBM tsv files
DBM_R_tsv_files = glob.glob(f"{working_dir}/metrics/DBM_stephanie_smoothed/sub-*_R.tsv")

# get a list of the R postivie DBM tsv files
DBM_R_positive_tsv_files = glob.glob(f"{working_dir}/metrics/DBM_stephanie_smoothed/sub-*_R_stru_conn.tsv")


In [29]:
# average the bilateral DBM tsv files
average_multiple_dfs(DBM_bilateral_tsv_files,
                     output_dir=f"{working_dir}/metrics/DBM_stephanie_smoothed",
                     output_tsv_stem="average_DBM_stephanie_smoothed_bilateral",
                     group_by_col="bilateral_region",
                     key_word="mean_voxel_value_bilateral")

average_multiple_dfs(DBM_R_tsv_files,
                     output_dir=f"{working_dir}/metrics/DBM_stephanie_smoothed",
                     output_tsv_stem="average_DBM_stephanie_smoothed_R",
                     group_by_col="name",
                     key_word="mean_voxel_value")

average_multiple_dfs(DBM_R_positive_tsv_files,
                     output_dir=f"{working_dir}/metrics/DBM_stephanie_smoothed",
                     output_tsv_stem="average_DBM_stephanie_smoothed_R_positive",
                     group_by_col="name",
                     key_word="mean_voxel_value")

Averaged tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/DBM_stephanie_smoothed/average_DBM_stephanie_smoothed_bilateral.tsv
Averaged tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/DBM_stephanie_smoothed/average_DBM_stephanie_smoothed_R.tsv
Averaged tsv saved to:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/BrainSMASH_correlations/metrics/DBM_stephanie_smoothed/average_DBM_stephanie_smoothed_R_positive.tsv


,name,mean_voxel_value
0,R_Somatomotor areas,-9.241761
1,R_Somatosensory areas,-10.820250
2,R_Auditory areas,-8.401207
3,R_Visual areas,-9.775158
4,R_Anterior cingulate area,-10.158166
5,R_Prelimbic area,-8.824630
6,R_Orbital area,-9.000368
7,R_Agranular insular area,-9.112457
8,R_Retrosplenial area,-9.516899
9,R_Posterior parietal association areas,-10.841655
